In [ ]:
# @title Setup: GitHub or an existing local checkout
from pathlib import Path
import os
import subprocess
import sys
import tempfile

USE_GITHUB = True  # @param {type:"boolean"}
USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
LOCAL_PATH = ""  # @param {type:"string"}
REPO_URL = "https://github.com/Baoshan-Song/KFV-FGO-Comparison.git"
BRANCH = "main"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if USE_GITHUB:
    # Every setup uses a fresh directory. Never delete a user's checkout.
    PROJECT_PATH = Path(tempfile.mkdtemp(prefix="kfv-fgo-"))/"repo"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, str(PROJECT_PATH)], check=True)
else:
    PROJECT_PATH = Path(LOCAL_PATH).expanduser().resolve() if LOCAL_PATH else Path.cwd()
    if not (PROJECT_PATH/"experiment_common.py").is_file():
        raise FileNotFoundError("Set LOCAL_PATH to the Python checkout")
os.chdir(PROJECT_PATH)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_PATH)], check=True)
# Switching checkouts on a repeated setup must not retain old modules.
for name in list(sys.modules):
    if name.split(".")[0] in {"core", "config", "data", "kfv_fgo", "experiment_common"}:
        del sys.modules[name]
sys.path.insert(0, str(PROJECT_PATH))
print("Using", PROJECT_PATH)
print("Re-FGO uses independent graph solves and actual second-stage QR.")


In [ ]:
# @title 1. KFV vs KFV
# @markdown Residuals show causal per-epoch estimates; iteration plots show the current epoch. Simulation parameters and both KFV estimator parameters are adjustable. Results play side-by-side below.
from IPython.display import display
import importlib
import experiment_common
importlib.reload(experiment_common)
from experiment_common import animate_pair, make_config, make_data, run_and_display

# --------------------------------------------------
# Simulation Data Parameters
# --------------------------------------------------
steps = 100  # @param {type:"slider", min:30, max:160, step:10}
distance = 105  # @param {type:"slider", min:100, max:1000, step:50}
seed = 7  # @param {type:"integer", min:0, max:999}
outlier_weight = 0.  # @param {type:"slider", min:0, max:1, step:0.05}
outlier_mean = 0.0  # @param {type:"number"}
outlier_sigma = 10.0  # @param {type:"number"}
white_sigma = 0.1  # @param {type:"number"}
initial_error_x = 100.0  # @param {type:"number"}
initial_error_y = -100.0  # @param {type:"number"}

# @markdown ---
# @markdown ### Left Column KFV Configuration

left_mode = "EKF"  # @param ["EKF", "iEKF", "rEKF", "riEKF"]

left_iterations = 2  # @param {type:"integer", min:1, max:20}
# @markdown *Note: Iterations are only used in iEKF and riEKF.*

left_kernel = "huber"  # @param ["none", "huber"]
# @markdown *Note: Robust kernel is only used in rEKF and riEKF.*

left_delta = 2.0  # @param {type:"number"}

# @markdown ---
# @markdown ### Right Column KFV Configuration

right_mode = "riEKF"  # @param ["EKF", "iEKF", "rEKF", "riEKF"]

right_iterations = 20  # @param {type:"integer", min:1, max:20}
# @markdown *Note: Iterations are only used in iEKF and riEKF.*

right_kernel = "huber"  # @param ["none", "huber"]
# @markdown *Note: Robust kernel is only used in rEKF and riEKF.*

right_delta = 2.0  # @param {type:"number"}

# --------------------------------------------------
# Helper Functions and Execution
# --------------------------------------------------
def make_ui_config(mode, iterations, kernel, delta):
  return make_config(kfv_mode=mode, max_iteration=iterations, robust_kernel=kernel, robust_delta=delta)

sim_data = make_data(seed=seed, num_steps=steps, anchor_radius=distance,
                     outlier_weight=outlier_weight, outlier_mean=outlier_mean,
                     outlier_sigma=outlier_sigma, white_sigma=white_sigma)

left = {"name": f"KFV-{left_mode}", "kind": "kfv", "config": make_ui_config(
    left_mode, left_iterations, left_kernel, left_delta)}

right = {"name": f"KFV-{right_mode}", "kind": "kfv", "config": make_ui_config(
    right_mode, right_iterations, right_kernel, right_delta)}

# Apply the displayed nominal measurement noise to both methods.
for item in (left, right):
    item["config"].kfv.r = item["config"].fgo.r = white_sigma**2
    for cfg in (item["config"].kfv, item["config"].fgo):
        cfg.err_x0[:2] = [initial_error_x, initial_error_y]
experiment = run_and_display(sim_data, left, right)
display(animate_pair(experiment, interval=80))

In [ ]:
# @title 2. KFV vs FGO
# @markdown Residuals show causal per-epoch estimates; iteration plots show the current epoch. Simulation parameters, KFV parameters, and FGO parameters are all adjustable. Results play side-by-side below.
from IPython.display import display
import importlib
import experiment_common
importlib.reload(experiment_common)
from experiment_common import animate_pair, make_config, make_data, run_and_display

# --------------------------------------------------
# Simulation Data Parameters
# --------------------------------------------------
steps = 100  # @param {type:"slider", min:30, max:160, step:10}
distance = 105  # @param {type:"slider", min:100, max:1000, step:50}
seed = 7  # @param {type:"integer", min:0, max:999}
outlier_weight = 0.2  # @param {type:"slider", min:0, max:1, step:0.05}
outlier_mean = 0.0  # @param {type:"number"}
outlier_sigma = 10.0  # @param {type:"number"}
white_sigma = 0.1  # @param {type:"number"}
initial_error_x = 100.0  # @param {type:"number"}
initial_error_y = -100.0  # @param {type:"number"}

# @markdown ---
# @markdown ### Left Column KFV Configuration

kfv_mode = "iEKF"  # @param ["EKF", "iEKF", "rEKF", "riEKF"]

kfv_iterations = 10  # @param {type:"integer", min:1, max:20}
# @markdown *Note: Iterations are only used in iEKF and riEKF.*

kfv_kernel = "none"  # @param ["none", "huber"]
# @markdown *Note: Robust kernel is only used in rEKF and riEKF.*

kfv_delta = 2.0  # @param {type:"number"}

# @markdown ---
# @markdown ### Right Column FGO Configuration

fgo_iterations = 10  # @param {type:"integer", min:1, max:20}

fgo_kernel = "huber"  # @param ["none", "huber"]

fgo_delta = 2.0  # @param {type:"number"}

fgo_imitate_kfv = True  # @param {type:"boolean"}
# @markdown *Note: Enabling imitate_kfv forces window_size=1.*

fgo_window_size = 1  # @param {type:"integer", min:1, max:20}
# @markdown Ordinary SW-FGO supports window 1, but is not the two-stage Re-FGO pipeline.

fgo_numerical_jacobian = False  # @param {type:"boolean"}

# @markdown Match the filter settings for an equivalence check. Turn off to explore independent FGO settings.
match_filter_settings = True  # @param {type:"boolean"}
# @markdown Numerical Jacobian uses central finite differences, not automatic differentiation.
from config.config import convert_kfv_to_fgo

# --------------------------------------------------
# Simulation & Execution
# --------------------------------------------------
sim_data = make_data(seed=seed, num_steps=steps, anchor_radius=distance,
                     outlier_weight=outlier_weight, outlier_mean=outlier_mean,
                     outlier_sigma=outlier_sigma, white_sigma=white_sigma)

left_config = make_config(
    kfv_mode=kfv_mode,
    max_iteration=kfv_iterations,
    robust_kernel=kfv_kernel,
    robust_delta=kfv_delta
)

if match_filter_settings:
    right_config = convert_kfv_to_fgo(left_config)
    right_config.fgo.autodiff = fgo_numerical_jacobian
    print("Matched Re-FGO: left filter's iteration, loss, noise and prior settings are used.")
else:
    right_config = make_config(max_iteration=fgo_iterations, robust_kernel=fgo_kernel,
        robust_delta=fgo_delta, imitate_kfv=fgo_imitate_kfv,
        window_size=1 if fgo_imitate_kfv else fgo_window_size,
        autodiff=fgo_numerical_jacobian)
    print("Independent configurations; equality is not expected.")

left = {"name": f"KFV-{kfv_mode}", "kind": "kfv", "config": left_config}
right = {"name": "Re-FGO" if right_config.fgo.imitate_kfv else "SW-FGO", "kind": "fgo", "config": right_config}

# Apply the displayed nominal measurement noise to both methods.
for item in (left, right):
    item["config"].kfv.r = item["config"].fgo.r = white_sigma**2
    for cfg in (item["config"].kfv, item["config"].fgo):
        cfg.err_x0[:2] = [initial_error_x, initial_error_y]
experiment = run_and_display(sim_data, left, right)
if match_filter_settings:
    import numpy as np
    a, b = [item["result"] for item in experiment["estimators"]]
    print("Maximum state difference:", np.max(np.abs(a["X_online"]-b["X_online"])))
    print("Maximum covariance difference:", np.max(np.abs(a["covariances"]-b["covariances"])))
display(animate_pair(experiment, interval=80))

In [ ]:
# @title 3. FGO vs FGO
# @markdown Residuals show causal per-epoch estimates; iteration plots show the current epoch. Simulation parameters and both FGO configurations are adjustable. Results play side-by-side below.
from IPython.display import display
import importlib
import experiment_common
importlib.reload(experiment_common)
from experiment_common import animate_pair, make_config, make_data, run_and_display

# --------------------------------------------------
# Simulation Data Parameters
# --------------------------------------------------
steps = 100  # @param {type:"slider", min:30, max:160, step:10}
distance = 105  # @param {type:"slider", min:100, max:1000, step:50}
seed = 7  # @param {type:"integer", min:0, max:999}
outlier_weight = 0.2  # @param {type:"slider", min:0, max:1, step:0.05}
outlier_mean = 0.0  # @param {type:"number"}
outlier_sigma = 10.0  # @param {type:"number"}
white_sigma = 0.1  # @param {type:"number"}
initial_error_x = 100.0  # @param {type:"number"}
initial_error_y = -100.0  # @param {type:"number"}

# @markdown ---
# @markdown ### Left Column FGO Configuration

left_iterations = 2  # @param {type:"integer", min:1, max:20}

left_kernel = "none"  # @param ["none", "huber"]

left_delta = 2.0  # @param {type:"number"}

left_imitate_kfv = False  # @param {type:"boolean"}
# @markdown *Note: Enabling imitate_kfv forces window_size=1.*

left_window_size = 2  # @param {type:"integer", min:1, max:20}
# @markdown Ordinary SW-FGO supports window 1, but is not the two-stage Re-FGO pipeline.

left_numerical_jacobian = False  # @param {type:"boolean"}

# @markdown ---
# @markdown ### Right Column FGO Configuration

right_iterations = 2  # @param {type:"integer", min:1, max:20}

right_kernel = "huber"  # @param ["none", "huber"]

right_delta = 2.0  # @param {type:"number"}

right_imitate_kfv = False  # @param {type:"boolean"}
# @markdown *Note: Enabling imitate_kfv forces window_size=1.*

right_window_size = 20  # @param {type:"integer", min:1, max:20}
# @markdown Ordinary SW-FGO supports window 1, but is not the two-stage Re-FGO pipeline.

right_numerical_jacobian = False  # @param {type:"boolean"}

# --------------------------------------------------
# Helper Functions and Execution
# --------------------------------------------------
def make_ui_config(iterations, kernel, delta, imitate, window, autodiff):
  return make_config(
      err_x=100.0, err_y=-100.0, err_vx=0.0, err_vy=0.0,
      p0_diag=(50.0, 50.0, 1.0, 1.0), kfv_mode="FGO",
      robust_kernel=kernel, robust_delta=delta, max_iteration=iterations,
      window_size=1 if imitate else window, imitate_kfv=imitate, autodiff=autodiff
  )

sim_data = make_data(seed=seed, num_steps=steps, anchor_radius=distance,
                     outlier_weight=outlier_weight, outlier_mean=outlier_mean,
                     outlier_sigma=outlier_sigma, white_sigma=white_sigma)

left = {"name": "FGO-config-1", "kind": "fgo", "config": make_ui_config(
    left_iterations, left_kernel, left_delta, left_imitate_kfv, left_window_size, left_numerical_jacobian)}

right = {"name": "FGO-config-2", "kind": "fgo", "config": make_ui_config(
    right_iterations, right_kernel, right_delta, right_imitate_kfv, right_window_size, right_numerical_jacobian)}

# Apply the displayed nominal measurement noise to both methods.
for item in (left, right):
    item["config"].kfv.r = item["config"].fgo.r = white_sigma**2
    for cfg in (item["config"].kfv, item["config"].fgo):
        cfg.err_x0[:2] = [initial_error_x, initial_error_y]
experiment = run_and_display(sim_data, left, right)
display(animate_pair(experiment, interval=80))